In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1ppHn2qkIvQFKg1NcWapyGYQNu7DzLylF", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/01_00_intro.mp3"))

In [ ]:
#@title 🎧 Listen: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Conversation Buffer and Sliding Window Memory

*Part 1 of the Vizuara series on Memory Architectures for LLM Applications*

**Estimated time: 50 minutes**

Every modern chatbot — ChatGPT, Claude, Gemini — gives the illusion of remembering your conversation. You say your name, mention your project, share your preferences, and the chatbot responds as if it knows you. But behind the scenes, something much simpler is happening: the entire conversation history is being stuffed into the prompt, every single time. In this notebook, we will build both the simplest version of this trick (the conversation buffer) and its first practical improvement (the sliding window), measure their costs, and see exactly where each one breaks down.

In [ ]:
#@title 🎧 Listen: Why Matter
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_02_why_matter.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 1. Why Does This Matter?

Imagine you are running a small restaurant. Every time a regular customer walks in, you hand them a notebook with every conversation you have ever had with them — every order, every complaint, every compliment — and say, "Please read all of this before we talk today."

On day one, the notebook has one page. Easy. On day thirty, it has thirty pages. A bit much, but manageable. On day three hundred, you are handing them a three-hundred-page tome just to order a sandwich. The customer is going to walk out.

**This is exactly what conversation buffer memory does.** It stores every message and sends everything to the LLM every time. It is simple, complete, and eventually catastrophic.

The sliding window is the obvious fix: instead of the entire notebook, you hand the customer only the last five pages. Fast, cheap, predictable — but now you have forgotten that they are allergic to peanuts, which they mentioned on page two.

In this notebook, we will:
- Build a **conversation buffer** from scratch and measure its token growth
- Build a **sliding window** and measure its cost ceiling
- **Visualize** the tradeoff between completeness and cost
- **Compare** both systems on a realistic conversation scenario
- Understand **exactly when** each approach fails

Let us begin.

In [ ]:
#@title 🎧 Listen: Intuition
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_03_intuition.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 2. Building Intuition

### The Stateless LLM Problem

Here is the fundamental thing to understand: **LLMs have no memory.** Every API call is completely independent. The model does not know who you are, what you said five minutes ago, or what it responded with last time. It is like calling a brilliant consultant who has amnesia — every call starts from zero.

So how does a chatbot appear to remember? The trick is brutally simple: you include the entire conversation history in every prompt.

```
Turn 1:
  Send: [System Prompt] + [User: "My name is Raj"]
  Receive: "Hello Raj! How can I help you?"

Turn 2:
  Send: [System Prompt] + [User: "My name is Raj"]
       + [Assistant: "Hello Raj! How can I help you?"]
       + [User: "I'm building a chatbot"]
  Receive: "That's exciting! What framework are you using?"

Turn 3:
  Send: [System Prompt] + [ALL previous messages] + [New message]
  Receive: Response
```

Do you see the problem? The input grows with every turn. By turn 50, you are sending tens of thousands of tokens — most of which the model has already "seen" and processed before. You are paying for the same information over and over.

### The Two Extremes

We can think of memory strategies on a spectrum:

- **Full buffer (keep everything):** Perfect recall, but cost grows linearly and eventually hits the context window ceiling.
- **No memory (keep nothing):** Zero cost, but the LLM forgets everything between turns.

The sliding window sits between these two extremes. It keeps the last $K$ turns and drops everything older. The question is: what is the right value of $K$? And what are we losing when we drop old messages?

Let us formalize this.

In [ ]:
#@title 🎧 Listen: Math Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_04_math_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 3. The Mathematics

### Token Cost of Buffer Memory

Let $t_i$ denote the total tokens at turn $i$ (user message + assistant response combined). The token cost of sending the full buffer at turn $n$ is:

$$C_{\text{buffer}}(n) = \sum_{i=1}^{n} t_i$$

This grows linearly. If each turn averages $\bar{t}$ tokens, then:

$$C_{\text{buffer}}(n) \approx n \cdot \bar{t}$$

**Worked example:** Suppose each turn averages 200 tokens. After 10 turns, we send $C(10) = 10 \times 200 = 2{,}000$ tokens. After 100 turns, $C(100) = 100 \times 200 = 20{,}000$ tokens. After 500 turns, $C(500) = 500 \times 200 = 100{,}000$ tokens. For a model with a 128K context window, we overflow around turn 640.

### Token Cost of Sliding Window

With a sliding window of size $K$ (keeping the last $K$ turns), the cost is bounded:

$$C_{\text{window}}(n) = \sum_{i=\max(1, n-K+1)}^{n} t_i \leq K \cdot \bar{t}$$

This is a constant ceiling, regardless of how long the conversation runs.

**Worked example:** With $K = 10$ and $\bar{t} = 200$ tokens per turn, $C_{\text{max}} = 10 \times 200 = 2{,}000$ tokens. Whether you are on turn 10 or turn 10,000, you never exceed 2,000 tokens of history. This is exactly what we want for cost predictability.

### Information Retention

But cost is only half the story. Let us define **information retention** as the fraction of total conversation tokens that are included in the context:

$$R(n) = \frac{C_{\text{context}}(n)}{C_{\text{total}}(n)}$$

For the buffer, $R(n) = 1$ (perfect retention) until the context window overflows. For the sliding window:

$$R_{\text{window}}(n) = \frac{\min(K, n) \cdot \bar{t}}{n \cdot \bar{t}} = \frac{\min(K, n)}{n}$$

**Worked example:** With $K = 10$, at turn 10 we have $R = 10/10 = 100\%$. At turn 50, $R = 10/50 = 20\%$. At turn 100, $R = 10/100 = 10\%$. We are forgetting 90% of the conversation. Any fact mentioned before the window is gone forever.

In [ ]:
#@title 🎧 Listen: Env Setup Heading
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_05_env_setup_heading.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Setting Up Our Environment

In [ ]:
#@title 🎧 Listen: Install Deps
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_06_install_deps.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Install dependencies (Colab-compatible, no GPU needed)
!pip install -q tiktoken numpy matplotlib

In [ ]:
#@title 🎧 Listen: Tokenizer Init
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_07_tokenizer_init.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
import tiktoken
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
from typing import List, Dict, Tuple, Optional
import time

# Initialize the tokenizer (GPT-4 encoding)
ENCODER = tiktoken.encoding_for_model("gpt-4")

def count_tokens(text: str) -> int:
    """Count tokens in a string using tiktoken."""
    return len(ENCODER.encode(text))

# Quick test
sample = "Hello, I am building a chatbot with long-term memory."
print(f"Text: '{sample}'")
print(f"Token count: {count_tokens(sample)}")
print(f"\nEnvironment ready!")

In [ ]:
#@title 🎧 Listen: Build Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_08_build_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 4. Let's Build It — Component by Component

### 4.1 The Conversation Buffer

Let us start with the simplest possible memory: store every message, send everything. We will track token usage at every step so we can measure the cost precisely.

In [ ]:
#@title 🎧 Listen: Buffer Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_09_buffer_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class ConversationBufferMemory:
    """Full conversation buffer — stores every message.

    This is the simplest memory strategy: keep everything,
    send everything. Token cost grows linearly with turns.
    """

    def __init__(self, system_prompt: str = ""):
        self.messages: List[Dict[str, str]] = []
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)
        self.token_history: List[int] = []

    def add_message(self, role: str, content: str):
        """Add a message to the buffer."""
        tokens = count_tokens(content)
        self.messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })
        total = self.get_total_tokens()
        self.token_history.append(total)
        return {
            "message_tokens": tokens,
            "total_tokens": total,
        }

    def get_total_tokens(self) -> int:
        """Total tokens across all stored messages."""
        return sum(m["tokens"] for m in self.messages)

    def get_context(self) -> str:
        """Build the full context string."""
        parts = []
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")
        for m in self.messages:
            parts.append(f"{m['role']}: {m['content']}")
        return "\n".join(parts)

    def get_context_tokens(self) -> int:
        """Tokens in the context we would send to the LLM."""
        return self.system_tokens + self.get_total_tokens()

print("ConversationBufferMemory defined!")

In [ ]:
#@title 🎧 Listen: Test Buffer Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_10_test_buffer_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us test it with a short conversation and watch the token count grow:

In [ ]:
#@title 🎧 Listen: Buffer Test Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_11_buffer_test_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
buffer = ConversationBufferMemory(
    system_prompt="You are a helpful AI assistant."
)

# A simple 5-turn conversation
turns = [
    ("user", "Hi! My name is Raj and I am building a chatbot."),
    ("assistant", "Hello Raj! That sounds like a great project. "
     "What kind of chatbot are you building?"),
    ("user", "It is a customer support bot for an e-commerce "
     "platform. Budget is $5,000."),
    ("assistant", "A customer support bot is a great use case. "
     "With a $5,000 budget, you have several options for "
     "the LLM backend and hosting."),
    ("user", "I want to use Python with FastAPI for the backend."),
    ("assistant", "Excellent choice. FastAPI is fast, async, and "
     "has great docs. For the LLM integration, you could "
     "use the OpenAI or Anthropic Python SDKs."),
    ("user", "The target audience is university students."),
    ("assistant", "University students will appreciate a clean, "
     "fast interface. Consider adding features like order "
     "tracking and FAQ search."),
    ("user", "Can you suggest a database for storing conversations?"),
    ("assistant", "PostgreSQL with pgvector is excellent for this. "
     "You get relational storage for conversations and "
     "vector search for semantic retrieval."),
]

print("Turn | Msg Tokens | Total History | Context Sent")
print("-" * 55)

for i, (role, content) in enumerate(turns):
    info = buffer.add_message(role, content)
    context_tokens = buffer.get_context_tokens()
    turn_num = i // 2 + 1 if i % 2 == 0 else ""
    print(f"  {str(turn_num):>2}  | {info['message_tokens']:>10} | "
          f"{info['total_tokens']:>13} | {context_tokens:>12}")

In [ ]:
#@title 🎧 Listen: Sliding Window Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_12_sliding_window_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Do you see it? After just 5 turns (10 messages), we are already using several hundred tokens of history. In a real conversation where responses are longer — say 300-500 tokens each — this grows much faster. Let us simulate a longer, more realistic conversation to see the full picture.

### 4.2 The Sliding Window

Now let us build the sliding window. The key difference: we keep all messages internally (for analysis), but the context we send to the LLM only includes the last $K$ turns.

In [ ]:
#@title 🎧 Listen: Window Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_13_window_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class SlidingWindowMemory:
    """Sliding window memory — keeps only the last K turns.

    Stores all messages internally for tracking, but
    only returns the most recent window_size turns
    as context for the LLM.
    """

    def __init__(
        self,
        window_size: int = 10,
        system_prompt: str = "",
    ):
        self.window_size = window_size
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)
        self.messages: List[Dict[str, str]] = []
        self.token_history: List[int] = []
        self.context_history: List[int] = []

    def add_message(self, role: str, content: str):
        """Add a message to the full history."""
        tokens = count_tokens(content)
        self.messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })
        total = sum(m["tokens"] for m in self.messages)
        self.token_history.append(total)

        # Also track the context window size
        ctx_tokens = self.get_context_tokens()
        self.context_history.append(ctx_tokens)

        return {
            "message_tokens": tokens,
            "total_tokens": total,
            "context_tokens": ctx_tokens,
        }

    def get_context_messages(self) -> List[Dict]:
        """Return only the messages within the window."""
        # Each turn = 2 messages (user + assistant)
        max_messages = self.window_size * 2
        if len(self.messages) <= max_messages:
            return list(self.messages)
        return list(self.messages[-max_messages:])

    def get_context(self) -> str:
        """Build the context string from the window."""
        parts = []
        if self.system_prompt:
            parts.append(f"System: {self.system_prompt}")
        for m in self.get_context_messages():
            parts.append(f"{m['role']}: {m['content']}")
        return "\n".join(parts)

    def get_context_tokens(self) -> int:
        """Tokens in the windowed context."""
        ctx = self.get_context_messages()
        return self.system_tokens + sum(
            m["tokens"] for m in ctx
        )

    def get_dropped_count(self) -> int:
        """Number of messages that have been dropped."""
        max_messages = self.window_size * 2
        return max(0, len(self.messages) - max_messages)

print("SlidingWindowMemory defined!")

In [ ]:
#@title 🎧 Listen: Test Window Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_14_test_window_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Let us test the sliding window with the same conversation and see how the context stays bounded:

In [ ]:
#@title 🎧 Listen: Window Test Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_15_window_test_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
window = SlidingWindowMemory(
    window_size=3,  # Keep only last 3 turns
    system_prompt="You are a helpful AI assistant.",
)

# Same 5-turn conversation
for role, content in turns:
    info = window.add_message(role, content)

print(f"Total messages stored: {len(window.messages)}")
print(f"Messages in window:   {len(window.get_context_messages())}")
print(f"Messages dropped:     {window.get_dropped_count()}")
print(f"\nTotal history tokens:  {window.token_history[-1]}")
print(f"Context window tokens: {window.get_context_tokens()}")
print(f"\n--- Messages in Context ---")
for m in window.get_context_messages():
    preview = m["content"][:60]
    print(f"  [{m['role']:>9}] {preview}...")

print(f"\n--- Dropped Messages ---")
max_msgs = window.window_size * 2
for m in window.messages[:len(window.messages) - max_msgs]:
    preview = m["content"][:60]
    print(f"  [{m['role']:>9}] {preview}...")

In [ ]:
#@title 🎧 Listen: Tradeoff Explanation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_16_tradeoff_explanation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Notice what happened. The user mentioned their name ("Raj"), their budget ("$5,000"), and their tech stack ("Python with FastAPI") in the first two turns. But with a window size of 3, those early messages have been dropped. If the user now asks "What budget did I mention?", the LLM will have no idea.

This is the fundamental tradeoff we need to understand deeply. Let us now simulate a much longer conversation to make this tradeoff visible.

In [ ]:
#@title 🎧 Listen: Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_17_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 5. Your Turn

### TODO 1: Implement a Token-Based Sliding Window

Our sliding window above uses a fixed number of turns. But turns can vary wildly in length — a one-word acknowledgment ("OK") versus a multi-paragraph explanation. A smarter approach is to set the window based on a **token budget** rather than a turn count.

Your task: implement `TokenBudgetWindow` that keeps as many recent messages as possible within a token budget.

In [ ]:
#@title 🎧 Listen: Todo1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_18_todo1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
class TokenBudgetWindow:
    """A sliding window that uses a token budget instead of
    a fixed turn count.

    Instead of keeping the last K turns, this keeps as many
    recent messages as fit within max_context_tokens.

    Args:
        max_context_tokens: Maximum tokens for the context
        system_prompt: The system prompt text

    Example:
        >>> win = TokenBudgetWindow(max_context_tokens=500)
        >>> win.add_message("user", "Hello!")
        >>> win.add_message("assistant", "Hi there!")
        >>> ctx = win.get_context_messages()
        >>> # Returns messages that fit in 500 tokens
    """

    def __init__(
        self,
        max_context_tokens: int = 2000,
        system_prompt: str = "",
    ):
        self.max_context_tokens = max_context_tokens
        self.system_prompt = system_prompt
        self.system_tokens = count_tokens(system_prompt)
        self.available_budget = (
            max_context_tokens - self.system_tokens
        )
        self.messages: List[Dict[str, str]] = []
        self.context_history: List[int] = []

    def add_message(self, role: str, content: str):
        """Add a message to history and track context size."""
        tokens = count_tokens(content)
        self.messages.append({
            "role": role,
            "content": content,
            "tokens": tokens,
        })
        ctx_tokens = self._compute_context_tokens()
        self.context_history.append(ctx_tokens)

    def get_context_messages(self) -> List[Dict]:
        """Return the most recent messages that fit
        within the token budget.

        Walk backwards from the most recent message.
        Keep adding messages as long as the running
        total does not exceed self.available_budget.
        Stop as soon as the next message would overflow.

        Returns:
            List of message dicts in chronological order
            (oldest first within the window).

        Hints:
            1. Initialize total_tokens = 0 and context = []
            2. Iterate over self.messages in REVERSE order
            3. For each message, check if adding it would
               exceed self.available_budget
            4. If it fits, prepend it to context (insert at 0)
            5. If it does not fit, stop
            6. Return context
        """
        # ---- YOUR CODE HERE ---- #

        context = []
        total_tokens = 0

        # TODO: Walk backwards through self.messages
        # TODO: Add messages that fit the budget
        # TODO: Return in chronological order

        return context

        # ---- END YOUR CODE ---- #

    def _compute_context_tokens(self) -> int:
        """Compute tokens in current context window."""
        ctx = self.get_context_messages()
        return self.system_tokens + sum(
            m["tokens"] for m in ctx
        )

In [ ]:
#@title 🎧 Listen: Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_19_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### TODO 2: Implement a Cost Comparison Function

Now let us build a function that simulates a conversation and compares the token costs of buffer vs sliding window memory across multiple turns.

In [ ]:
#@title 🎧 Listen: Todo2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_20_todo2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
def compare_memory_costs(
    n_turns: int = 50,
    avg_user_tokens: int = 50,
    avg_assistant_tokens: int = 150,
    window_sizes: List[int] = None,
    seed: int = 42,
) -> Dict:
    """Simulate a conversation and compare token costs
    for buffer memory vs multiple sliding window sizes.

    Args:
        n_turns: Number of conversation turns to simulate
        avg_user_tokens: Average tokens per user message
        avg_assistant_tokens: Average tokens per assistant msg
        window_sizes: List of K values for sliding windows
            (default: [5, 10, 20])
        seed: Random seed for reproducibility

    Returns:
        Dict with keys:
            'turns': list of turn numbers [1, 2, ..., n_turns]
            'buffer_costs': list of cumulative token costs
                for buffer memory at each turn
            'window_costs': dict mapping window_size to list
                of context token costs at each turn
            'total_tokens': total tokens generated across
                all turns (for reference)

    Steps:
        1. Set numpy random seed
        2. Generate random token counts for each turn:
           - user_tokens = max(10, normal(avg_user, avg_user*0.3))
           - asst_tokens = max(20, normal(avg_asst, avg_asst*0.3))
        3. For each turn, compute:
           - Buffer cost: cumulative sum of all turn tokens
           - Window cost for each K: sum of last K turns'
             tokens (or all turns if fewer than K)
        4. Return the results dict

    Example:
        >>> result = compare_memory_costs(n_turns=10)
        >>> len(result['turns'])
        10
        >>> result['buffer_costs'][-1] > result['window_costs'][5][-1]
        True
    """
    if window_sizes is None:
        window_sizes = [5, 10, 20]

    # ---- YOUR CODE HERE ---- #

    turns = list(range(1, n_turns + 1))
    buffer_costs = []
    window_costs = {k: [] for k in window_sizes}

    np.random.seed(seed)

    # TODO: Generate token counts per turn
    turn_tokens = []  # list of (user_tok, asst_tok) per turn

    # TODO: Compute buffer costs (cumulative sum)

    # TODO: Compute window costs for each K

    return {
        "turns": turns,
        "buffer_costs": buffer_costs,
        "window_costs": window_costs,
        "total_tokens": 0,  # TODO: compute total
    }

    # ---- END YOUR CODE ---- #

In [ ]:
#@title 🎧 Listen: Verify Todo1 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_21_verify_todo1_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 1

In [ ]:
# --- Test TokenBudgetWindow ---
tbw = TokenBudgetWindow(
    max_context_tokens=300,
    system_prompt="You are helpful.",
)

# Add messages of varying lengths
test_msgs = [
    ("user", "Short message."),
    ("assistant", "A slightly longer response with more detail."),
    ("user", "Another short one."),
    ("assistant", "This is a much longer response that goes on "
     "and on with lots of detail about the topic at hand, "
     "providing examples, explanations, and context that "
     "the user might find helpful for their question."),
    ("user", "Final question about the project."),
]

for role, content in test_msgs:
    tbw.add_message(role, content)

ctx = tbw.get_context_messages()
ctx_tokens = sum(m["tokens"] for m in ctx)

print(f"Total messages stored: {len(tbw.messages)}")
print(f"Messages in context: {len(ctx)}")
print(f"Context tokens: {ctx_tokens}")
print(f"Budget: {tbw.available_budget}")

assert ctx_tokens <= tbw.available_budget, \
    f"Context tokens ({ctx_tokens}) exceed budget ({tbw.available_budget})"
assert len(ctx) > 0, "Context should not be empty"
assert len(ctx) <= len(tbw.messages), "Cannot have more context than messages"

# Check that messages are in chronological order
if len(ctx) > 1:
    orig_indices = [
        tbw.messages.index(m) for m in ctx
    ]
    assert orig_indices == sorted(orig_indices), \
        "Context messages must be in chronological order"


In [ ]:
#@title 🎧 Listen: Verify Todo1 Code After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_22_verify_todo1_code_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("\nAll TokenBudgetWindow tests passed!")

In [ ]:
#@title 🎧 Listen: Verify Todo2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_23_verify_todo2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Verification Cell for TODO 2

In [ ]:
# --- Test compare_memory_costs ---
result = compare_memory_costs(
    n_turns=20, window_sizes=[5, 10]
)

assert len(result["turns"]) == 20
assert len(result["buffer_costs"]) == 20
assert 5 in result["window_costs"]
assert 10 in result["window_costs"]
assert len(result["window_costs"][5]) == 20

# Buffer should always be >= window
for i in range(20):
    assert result["buffer_costs"][i] >= result["window_costs"][5][i], \
        f"Buffer should cost >= window at turn {i+1}"

# Window K=10 should be >= window K=5
for i in range(20):
    assert result["window_costs"][10][i] >= result["window_costs"][5][i], \
        f"Larger window should cost more at turn {i+1}"

# Buffer cost should be monotonically increasing
for i in range(1, 20):
    assert result["buffer_costs"][i] >= result["buffer_costs"][i-1], \
        "Buffer cost must be monotonically increasing"


In [ ]:
#@title 🎧 Listen: Verify Todo2 Code After
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_24_verify_todo2_code_after.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("All compare_memory_costs tests passed!")

In [ ]:
#@title 🎧 Listen: Solutions Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_25_solutions_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

<details>
<summary><b>Solution: TokenBudgetWindow.get_context_messages</b></summary>

In [ ]:
def get_context_messages(self) -> List[Dict]:
    context = []
    total_tokens = 0

    for msg in reversed(self.messages):
        if total_tokens + msg["tokens"] > self.available_budget:
            break
        context.insert(0, msg)
        total_tokens += msg["tokens"]

    return context

</details>

<details>
<summary><b>Solution: compare_memory_costs</b></summary>

In [ ]:
def compare_memory_costs(
    n_turns=50, avg_user_tokens=50,
    avg_assistant_tokens=150, window_sizes=None, seed=42,
):
    if window_sizes is None:
        window_sizes = [5, 10, 20]

    np.random.seed(seed)
    turns = list(range(1, n_turns + 1))
    buffer_costs = []
    window_costs = {k: [] for k in window_sizes}

    turn_tokens = []
    for _ in range(n_turns):
        u = max(10, int(np.random.normal(
            avg_user_tokens, avg_user_tokens * 0.3)))
        a = max(20, int(np.random.normal(
            avg_assistant_tokens, avg_assistant_tokens * 0.3)))
        turn_tokens.append(u + a)

    cumulative = 0
    for i, tok in enumerate(turn_tokens):
        cumulative += tok
        buffer_costs.append(cumulative)

        for k in window_sizes:
            start = max(0, i + 1 - k)
            w_cost = sum(turn_tokens[start:i + 1])
            window_costs[k].append(w_cost)

    return {
        "turns": turns,
        "buffer_costs": buffer_costs,
        "window_costs": window_costs,
        "total_tokens": cumulative,
    }

</details>

In [ ]:
#@title 🎧 Listen: Run Solutions
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_26_run_solutions.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

Now let us run the solutions and proceed:

In [ ]:
#@title 🎧 Listen: Load Solutions Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_27_load_solutions_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# --- Solution: TokenBudgetWindow ---
class TokenBudgetWindow(TokenBudgetWindow):
    def get_context_messages(self) -> List[Dict]:
        context = []
        total_tokens = 0
        for msg in reversed(self.messages):
            if total_tokens + msg["tokens"] > self.available_budget:
                break
            context.insert(0, msg)
            total_tokens += msg["tokens"]
        return context

# --- Solution: compare_memory_costs ---
def compare_memory_costs(
    n_turns=50, avg_user_tokens=50,
    avg_assistant_tokens=150, window_sizes=None, seed=42,
):
    if window_sizes is None:
        window_sizes = [5, 10, 20]

    np.random.seed(seed)
    turns_list = list(range(1, n_turns + 1))
    buffer_costs = []
    window_costs = {k: [] for k in window_sizes}

    turn_tokens = []
    for _ in range(n_turns):
        u = max(10, int(np.random.normal(
            avg_user_tokens, avg_user_tokens * 0.3)))
        a = max(20, int(np.random.normal(
            avg_assistant_tokens, avg_assistant_tokens * 0.3)))
        turn_tokens.append(u + a)

    cumulative = 0
    for i, tok in enumerate(turn_tokens):
        cumulative += tok
        buffer_costs.append(cumulative)
        for k in window_sizes:
            start = max(0, i + 1 - k)
            w_cost = sum(turn_tokens[start:i + 1])
            window_costs[k].append(w_cost)

    return {
        "turns": turns_list,
        "buffer_costs": buffer_costs,
        "window_costs": window_costs,
        "total_tokens": cumulative,
    }

print("Solutions loaded!")

In [ ]:
#@title 🎧 Listen: Putting Together Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_28_putting_together_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 6. Putting It All Together

Let us now run both memory systems on a realistic 50-turn conversation and see the full picture. We will simulate a technical consulting conversation where the user discusses multiple topics over time.

In [ ]:
#@title 🎧 Listen: Simulate Conversation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_29_simulate_conversation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Simulate a 50-turn conversation
np.random.seed(42)

topics = [
    "project requirements", "database design",
    "API architecture", "authentication", "deployment",
    "monitoring", "testing", "performance tuning",
    "error handling", "documentation",
]

system_prompt = "You are a senior software architect."

# Create both memory systems
buf_mem = ConversationBufferMemory(system_prompt=system_prompt)
win_mem_5 = SlidingWindowMemory(window_size=5, system_prompt=system_prompt)
win_mem_10 = SlidingWindowMemory(window_size=10, system_prompt=system_prompt)
win_mem_20 = SlidingWindowMemory(window_size=20, system_prompt=system_prompt)

for i in range(50):
    topic = topics[i % len(topics)]

    # Generate realistic messages
    user_len = max(20, int(np.random.normal(60, 20)))
    asst_len = max(40, int(np.random.normal(150, 50)))

    user_msg = f"I have a question about {topic}. " + "x " * user_len
    asst_msg = (
        f"Regarding {topic}, here are my recommendations. "
        + "y " * asst_len
    )

    buf_mem.add_message("user", user_msg)
    buf_mem.add_message("assistant", asst_msg)
    win_mem_5.add_message("user", user_msg)
    win_mem_5.add_message("assistant", asst_msg)
    win_mem_10.add_message("user", user_msg)
    win_mem_10.add_message("assistant", asst_msg)
    win_mem_20.add_message("user", user_msg)
    win_mem_20.add_message("assistant", asst_msg)

print(f"Conversation simulated: 50 turns")
print(f"\nBuffer memory:   {buf_mem.get_total_tokens():>6} tokens")
print(f"Window K=5:      {win_mem_5.get_context_tokens():>6} tokens")
print(f"Window K=10:     {win_mem_10.get_context_tokens():>6} tokens")
print(f"Window K=20:     {win_mem_20.get_context_tokens():>6} tokens")

In [ ]:
#@title 🎧 Listen: Viz Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_30_viz_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 7. Training and Results

Now let us visualize the results. We will create three charts that reveal the cost-memory tradeoff clearly.

### Visualization 1: Token Cost Over Time

In [ ]:
#@title 🎧 Listen: Viz1 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_31_viz1_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Absolute token costs
ax1 = axes[0]
turns_x = list(range(1, 51))

# Buffer cost (cumulative)
ax1.plot(turns_x, buf_mem.token_history[::2],
         color="#EF4444", linewidth=2.5,
         label="Buffer (full history)")

# Window costs
colors = {"K=5": "#10B981", "K=10": "#3B82F6", "K=20": "#8B5CF6"}
for mem, label in [(win_mem_5, "K=5"), (win_mem_10, "K=10"),
                    (win_mem_20, "K=20")]:
    # Take every other entry (after assistant message)
    ctx_vals = mem.context_history[1::2]
    ax1.plot(turns_x[:len(ctx_vals)], ctx_vals,
             color=colors[label], linewidth=2,
             linestyle="--", label=f"Window {label}")

ax1.axhline(y=8192, color="gray", linestyle=":",
            alpha=0.5, label="8K context limit")
ax1.set_xlabel("Conversation Turn", fontsize=12)
ax1.set_ylabel("Tokens Sent to LLM", fontsize=12)
ax1.set_title("Token Cost: Buffer vs Sliding Window",
              fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Right: Information retention
ax2 = axes[1]
buffer_total = buf_mem.token_history[1::2]
for mem, label in [(win_mem_5, "K=5"), (win_mem_10, "K=10"),
                    (win_mem_20, "K=20")]:
    ctx_vals = mem.context_history[1::2]
    retention = [
        c / t * 100 if t > 0 else 100
        for c, t in zip(ctx_vals, buffer_total)
    ]
    ax2.plot(turns_x[:len(retention)], retention,
             color=colors[label], linewidth=2.5,
             label=f"Window {label}")

ax2.axhline(y=100, color="#EF4444", linewidth=2,
            label="Buffer (100%)")
ax2.axhline(y=50, color="gray", linestyle=":",
            alpha=0.5)
ax2.text(51, 51, "50%", fontsize=9, color="gray")
ax2.set_xlabel("Conversation Turn", fontsize=12)
ax2.set_ylabel("Information Retention (%)", fontsize=12)
ax2.set_title("Information Retention Over Time",
              fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.set_ylim(0, 110)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("buffer_vs_window_cost.png", dpi=150,
            bbox_inches="tight")
plt.show()

print("Left: Buffer cost grows linearly; windows stay flat.")
print("Right: Windows lose information as the conversation grows.")

In [ ]:
#@title 🎧 Listen: Viz2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_32_viz2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 2: Cost Per Dollar

Let us translate token counts into actual API costs to make this tangible.

In [ ]:
#@title 🎧 Listen: Viz2 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_33_viz2_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
#@title 🎧 Listen: Viz3 Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_35_viz3_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
# Pricing (approximate, per 1K input tokens)
PRICE_PER_1K = {
    "GPT-4": 0.03,
    "GPT-4 Turbo": 0.01,
    "Claude 3 Sonnet": 0.003,
    "Gemini 1.5 Flash": 0.000075,
}

fig, ax = plt.subplots(figsize=(14, 6))

# Use the buffer memory token history
buffer_costs_per_turn = buf_mem.token_history[1::2]
window_5_costs = win_mem_5.context_history[1::2]

x = list(range(1, len(buffer_costs_per_turn) + 1))

bar_width = 0.35
models = list(PRICE_PER_1K.keys())
model_colors = ["#EF4444", "#F59E0B", "#3B82F6", "#10B981"]

# Show total cost for a 50-turn conversation
total_buffer = sum(buffer_costs_per_turn)
total_window = sum(window_5_costs)

buffer_dollars = [
    total_buffer / 1000 * p for p in PRICE_PER_1K.values()
]
window_dollars = [
    total_window / 1000 * p for p in PRICE_PER_1K.values()
]

x_pos = np.arange(len(models))
bars1 = ax.bar(x_pos - bar_width/2, buffer_dollars,
               bar_width, label="Buffer (full history)",
               color="#EF4444", alpha=0.8)
bars2 = ax.bar(x_pos + bar_width/2, window_dollars,
               bar_width, label="Window K=5",
               color="#10B981", alpha=0.8)

# Add value labels
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
            f"${h:.3f}", ha="center", fontsize=9,
            fontweight="bold")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
            f"${h:.3f}", ha="center", fontsize=9,
            fontweight="bold")

ax.set_xticks(x_pos)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel("Total Cost (USD) for 50-Turn Conversation",
              fontsize=12)
ax.set_title("API Cost: Buffer vs Window K=5 "
             "(50-Turn Conversation)",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("cost_comparison.png", dpi=150,
            bbox_inches="tight")
plt.show()

savings_pct = (1 - total_window / total_buffer) * 100
print(f"\nTotal input tokens — Buffer: {total_buffer:,}")
print(f"Total input tokens — Window K=5: {total_window:,}")
print(f"Token savings: {savings_pct:.1f}%")

In [ ]:
#@title 🎧 Listen: Viz3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_34_viz3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

### Visualization 3: The Forgetting Frontier

Let us visualize exactly which information is lost by the sliding window at each turn.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

n_turns = 30
K = 5

# Create a heatmap showing which turns are in context
heatmap = np.zeros((n_turns, n_turns))

for current_turn in range(n_turns):
    start = max(0, current_turn - K + 1)
    for t in range(start, current_turn + 1):
        heatmap[current_turn, t] = 1.0

im = ax.imshow(heatmap, cmap="Blues", aspect="auto",
               interpolation="nearest")

ax.set_xlabel("Message Turn (original position)",
              fontsize=12)
ax.set_ylabel("Current Turn (when context is built)",
              fontsize=12)
ax.set_title(f"Sliding Window Heatmap (K={K}): "
             f"Which Turns Are in Context?",
             fontsize=14, fontweight="bold")

# Add annotation
ax.annotate("Blue = in context\nWhite = forgotten",
            xy=(0.02, 0.95), xycoords="axes fraction",
            fontsize=10, verticalalignment="top",
            bbox=dict(boxstyle="round,pad=0.3",
                      facecolor="white", alpha=0.8))

plt.colorbar(im, ax=ax, label="In Context (1) or Not (0)")
plt.tight_layout()
plt.savefig("forgetting_frontier.png", dpi=150,
            bbox_inches="tight")
plt.show()

print(f"The diagonal band shows the sliding window.")
print(f"Everything below and to the left is forgotten.")
print(f"At turn 20, only turns 16-20 are in context.")

In [ ]:
#@title 🎧 Listen: Viz Summary
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_36_viz_summary.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

These visualizations tell a clear story:

1. **Buffer cost grows without bound** — it is a ticking time bomb that will eventually hit the context window ceiling or drain your API budget.
2. **Sliding windows are cost-predictable** but aggressively forgetful — with K=5, you lose 90% of the conversation by turn 50.
3. **The forgetting is total** — once a message slides out of the window, it is gone. There is no partial recall, no "fuzzy memory." It is binary: in the window or forgotten.

This is why we need smarter memory strategies. In the next notebook, we will explore summary memory — compressing old messages instead of dropping them entirely.

In [ ]:
#@title 🎧 Listen: Final Demo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_37_final_demo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 8. Final Output

Let us run a complete demonstration that shows both systems handling a realistic scenario: the user mentions important facts early, discusses other topics, and then asks about the earlier facts.

In [ ]:
#@title 🎧 Listen: Final Demo Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_38_final_demo_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("=" * 60)
print("  FINAL DEMO: THE FORGETTING PROBLEM")
print("=" * 60)

# Build both memories
demo_buffer = ConversationBufferMemory(
    system_prompt="You are a project planning assistant."
)
demo_window = SlidingWindowMemory(
    window_size=3,
    system_prompt="You are a project planning assistant.",
)

# Phase 1: Important facts (turns 1-2)
phase1 = [
    ("user", "My name is Raj. The project budget is $5,000 "
     "and the deadline is March 15."),
    ("assistant", "Got it, Raj! I'll keep the $5,000 budget "
     "and March 15 deadline in mind. What would you like "
     "to plan first?"),
    ("user", "The team has 4 people: me, Sarah (frontend), "
     "Alex (backend), and Priya (ML)."),
    ("assistant", "Great team of 4! With Raj, Sarah on "
     "frontend, Alex on backend, and Priya on ML, you "
     "have good coverage. What is the project about?"),
]

# Phase 2: Filler conversation (turns 3-5)
phase2 = [
    ("user", "What tools do you recommend for project management?"),
    ("assistant", "I recommend Jira for task tracking, "
     "Notion for documentation, and Slack for communication."),
    ("user", "What about CI/CD pipelines?"),
    ("assistant", "GitHub Actions is great for CI/CD. "
     "Set up automated testing and deployment."),
    ("user", "Should we use Docker for deployment?"),
    ("assistant", "Yes, Docker with Docker Compose for "
     "local development, and Kubernetes for production "
     "if you need to scale."),
]

# Phase 3: The callback question
phase3 = [
    ("user", "Remind me — what was our budget and deadline?"),
]

# Add all messages to both systems
for role, content in phase1 + phase2 + phase3:
    demo_buffer.add_message(role, content)
    demo_window.add_message(role, content)

# Show what each system knows at query time
print("\n--- QUERY: 'What was our budget and deadline?' ---\n")

# Buffer context
print("BUFFER MEMORY (full history):")
print(f"  Context tokens: {demo_buffer.get_context_tokens()}")
print(f"  Contains budget info: YES")
print(f"  Contains deadline info: YES")
print(f"  Contains team info: YES")
budget_in_buffer = any(
    "$5,000" in m["content"] for m in demo_buffer.messages
)
print(f"  '$5,000' found in context: {budget_in_buffer}")

print()

# Window context
print(f"SLIDING WINDOW (K={demo_window.window_size}):")
print(f"  Context tokens: {demo_window.get_context_tokens()}")
ctx_messages = demo_window.get_context_messages()
budget_in_window = any(
    "$5,000" in m["content"] for m in ctx_messages
)
deadline_in_window = any(
    "March 15" in m["content"] for m in ctx_messages
)
team_in_window = any(
    "Sarah" in m["content"] for m in ctx_messages
)
print(f"  Contains budget info: {'YES' if budget_in_window else 'NO -- LOST!'}")
print(f"  Contains deadline info: {'YES' if deadline_in_window else 'NO -- LOST!'}")
print(f"  Contains team info: {'YES' if team_in_window else 'NO -- LOST!'}")

print(f"\n  Messages in window:")
for m in ctx_messages:
    preview = m["content"][:65]
    print(f"    [{m['role']:>9}] {preview}...")

print(f"\n  Messages DROPPED (forgotten):")
max_msgs = demo_window.window_size * 2
dropped = demo_window.messages[:len(demo_window.messages) - max_msgs]
for m in dropped:
    preview = m["content"][:65]
    has_fact = any(
        kw in m["content"]
        for kw in ["$5,000", "March 15", "Sarah", "Raj"]
    )
    marker = " << IMPORTANT FACT LOST" if has_fact else ""
    print(f"    [{m['role']:>9}] {preview}...{marker}")

# Summary stats
print(f"\n{'=' * 60}")
print(f"Buffer: {demo_buffer.get_context_tokens()} tokens, "
      f"perfect recall")
print(f"Window: {demo_window.get_context_tokens()} tokens, "
      f"critical facts LOST")
print(f"Token savings: "
      f"{(1 - demo_window.get_context_tokens() / demo_buffer.get_context_tokens()) * 100:.0f}%")
print(f"Information cost: budget, deadline, and team "
      f"names are gone")
print(f"{'=' * 60}")

In [ ]:
#@title 🎧 Listen: Reflection
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_39_reflection.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

## 9. Reflection and Next Steps

### What Did We Learn?

We built two memory systems from scratch and measured their behavior precisely:

1. **Conversation buffer** is simple and lossless, but its cost grows linearly — $C(n) = n \cdot \bar{t}$. It will eventually overflow any context window.

2. **Sliding window** has a bounded cost of $C_{\text{max}} = K \cdot \bar{t}$, but its information retention drops to $K/n$ as the conversation grows. Critical early facts are lost permanently.

3. **The core tradeoff is cost vs completeness.** There is no free lunch — you either pay in tokens (buffer) or pay in forgotten information (window).

4. **Token-based budgeting** (our TODO 1) is strictly better than turn-based windowing because it adapts to variable message lengths.

### Questions to Think About

1. Is there a way to keep the cost bounded (like the window) but still retain important early facts? What would that system look like?

2. What if we could **compress** old messages instead of dropping them? How much information could we preserve in, say, 300 tokens of compressed history?

3. The sliding window treats all messages equally — a one-word "OK" takes the same "slot" as a paragraph of critical requirements. How would you prioritize which messages to keep?

4. In a production system, would you let the user configure the window size? Or would you make it adaptive based on conversation content?

5. We measured token cost, but what about **latency**? Larger contexts mean slower inference. How would you factor time-to-first-token into your memory strategy?

### What Comes Next

The sliding window solves the cost problem but creates a new one: information loss. In the next notebook, we will explore **summary memory** — using the LLM itself to compress old messages into a running summary. This gives us bounded cost *and* information retention. The key question will be: how much information survives the compression?

In [ ]:
#@title 🎧 Listen: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/01_40_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")

In [ ]:
print("Notebook complete!")
print()
print("Key equations to remember:")
print("  Buffer cost:  C(n) = n * t_avg")
print("  Window cost:  C_max = K * t_avg")
print("  Retention:    R(n) = min(K, n) / n")
print()
print("The fundamental tradeoff: cost vs completeness.")
print("Next up: Summary Memory — compressing instead of forgetting.")